In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path("C:/Users/ACER/Downloads/New folder/cleaned_datasets")

collision_df = pd.read_csv(data_path / "collision_cleaned.csv", dtype={"collision_index": str})
vehicle_df = pd.read_csv(data_path / "vehicle_cleaned.csv", dtype={"collision_index": str})
casualty_df = pd.read_csv(data_path / "casualty_cleaned.csv", dtype={"collision_index": str})

print("Collision:", collision_df.shape)
print("Vehicle  :", vehicle_df.shape)
print("Casualty :", casualty_df.shape)

C:\Users\ACER\AppData\Local\Temp\ipykernel_18440\1282267685.py:7: DtypeWarning: Columns (2: collision_ref_no) have mixed types. Specify dtype option on import or set low_memory=False.
  collision_df = pd.read_csv(data_path / "collision_cleaned.csv", dtype={"collision_index": str})
C:\Users\ACER\AppData\Local\Temp\ipykernel_18440\1282267685.py:8: DtypeWarning: Columns (27: generic_make_model) have mixed types. Specify dtype option on import or set low_memory=False.
  vehicle_df = pd.read_csv(data_path / "vehicle_cleaned.csv", dtype={"collision_index": str})


Collision: (513801, 44)
Vehicle  : (937265, 32)
Casualty : (652821, 24)


Task 8 — Aggregate VEHICLE to Collision Level
8.1 Inspect Vehicle Type Distribution

Before creating has_car, has_motorcycle, and similar flags, check which vehicle_type codes actually appear in the data and at what frequency. This ensures the bucket groupings reflect real data rather than the full theoretical code list, and flags any rare categories that may need to be merged into an "Other" bucket rather than given their own flag.

In [4]:
vehicle_type_counts = vehicle_df["vehicle_type"].value_counts(dropna=False).sort_index()
vehicle_type_pct = vehicle_df["vehicle_type"].value_counts(normalize=True, dropna=False).sort_index() * 100

vehicle_type_summary = pd.DataFrame({
    "Count": vehicle_type_counts,
    "Percentage": vehicle_type_pct.round(2)
})

print("Vehicle Type distribution (raw codes)")
print("-" * 50)
display(vehicle_type_summary)

Vehicle Type distribution (raw codes)
--------------------------------------------------


,Count,Percentage
vehicle_type,,
-1,661,0.07
1,81486,8.69
2,5008,0.53
3,44795,4.78
4,9590,1.02
5,23011,2.46
8,15362,1.64
9,637681,68.04
10,1506,0.16


8.2 Define Vehicle-Type Buckets and Create Flags

Based on the actual code distribution, vehicle types are grouped into meaningful buckets: car (including taxis), motorcycle (all cc variants), pedal cycle, bus/coach, van/light goods, and heavy goods (HGV). Rare categories (horse, agricultural vehicle, tram, mobility scooter, other) are grouped into an "other" bucket. Code 33, which does not appear in the official STATS19 data guide, is flagged separately as undocumented rather than silently merged into "other."

In [5]:
def vehicle_bucket(code):
    if code == 1:
        return "pedal_cycle"
    elif code in [2, 3, 4, 5, 23, 97, 103, 104, 105, 106]:
        return "motorcycle"
    elif code in [8, 9, 108, 109]:
        return "car"
    elif code in [10, 11, 110]:
        return "bus_or_coach"
    elif code == 19:
        return "van_or_goods_light"
    elif code in [20, 21, 98, 113]:
        return "hgv"
    elif code == 33:
        return "undocumented_code_33"
    elif code in [16, 17, 18, 22, 90, 99, -1]:
        return "other"
    else:
        return "other"

vehicle_df["vehicle_bucket"] = vehicle_df["vehicle_type"].apply(vehicle_bucket)

bucket_counts = vehicle_df["vehicle_bucket"].value_counts()
print("Vehicle bucket distribution")
print("-" * 40)
print(bucket_counts)

Vehicle bucket distribution
----------------------------------------
vehicle_bucket
car                     653043
motorcycle               87294
pedal_cycle              81486
van_or_goods_light       60309
hgv                      21213
bus_or_coach             16606
other                    10413
undocumented_code_33      6901
Name: count, dtype: int64


8.3 Aggregate Vehicle Buckets to Collision Level

Group Vehicle records by collision_index and create one binary flag per bucket (has_car, has_motorcycle, etc.), a vehicle_type_diversity count of distinct buckets involved, and driver-age aggregates (min_driver_age, max_driver_age, avg_driver_age) computed only from available (non-missing) values. The result is one row per collision.

In [6]:
# One-hot the bucket column, then aggregate to collision level
bucket_dummies = pd.get_dummies(vehicle_df["vehicle_bucket"])
bucket_dummies["collision_index"] = vehicle_df["collision_index"]

# has_* flags: 1 if at least one vehicle of that bucket is present
vehicle_flags = bucket_dummies.groupby("collision_index").max().astype(int)
vehicle_flags.columns = [f"has_{col}" for col in vehicle_flags.columns]

# vehicle_type_diversity: count of distinct buckets present per collision
vehicle_diversity = (
    vehicle_df.groupby("collision_index")["vehicle_bucket"]
    .nunique()
    .rename("vehicle_type_diversity")
)

# Driver age aggregates (age_of_driver already has -1 converted to NaN from notebook 05)
driver_age_agg = (
    vehicle_df.groupby("collision_index")["age_of_driver"]
    .agg(min_driver_age="min", max_driver_age="max", avg_driver_age="mean")
)

# Combine into one Vehicle aggregation table
vehicle_agg = vehicle_flags.join(vehicle_diversity).join(driver_age_agg).reset_index()

print("Vehicle aggregation shape:", vehicle_agg.shape)
print("Unique collision_index in Vehicle:", vehicle_df["collision_index"].nunique())
vehicle_agg.head()

Vehicle aggregation shape: (513801, 13)
Unique collision_index in Vehicle: 513801


,collision_index,has_bus_or_coach,has_car,has_hgv,has_motorcycle,has_other,has_pedal_cycle,has_undocumented_code_33,has_van_or_goods_light,vehicle_type_diversity,min_driver_age,max_driver_age,avg_driver_age
0,2021010287148,0,1,0,0,0,0,0,0,1,19.0,19.0,19.0
1,2021010287149,0,1,0,0,0,0,0,0,1,21.0,32.0,26.5
2,2021010287151,0,1,0,0,0,0,0,0,1,27.0,27.0,27.0
3,2021010287155,0,1,0,0,0,0,0,0,1,NaN,NaN,NaN
4,2021010287157,0,1,0,0,0,0,0,0,1,37.0,37.0,37.0


8.4 Validate Vehicle Aggregation Integrity

Confirm the aggregation produced exactly one row per unique collision (no row multiplication or loss), and sanity-check the diversity and age ranges for plausibility before joining this table to Collision.

In [7]:
print("Vehicle aggregation row count:", vehicle_agg.shape[0])
print("Unique collision_index in raw Vehicle table:", vehicle_df["collision_index"].nunique())
print("Match:", vehicle_agg.shape[0] == vehicle_df["collision_index"].nunique())

print("\nvehicle_type_diversity range:", vehicle_agg["vehicle_type_diversity"].min(), "to", vehicle_agg["vehicle_type_diversity"].max())
print("\nDriver age sanity check:")
print(vehicle_agg[["min_driver_age", "max_driver_age", "avg_driver_age"]].describe())

print("\nAny duplicate collision_index in vehicle_agg?", vehicle_agg["collision_index"].duplicated().sum())

Vehicle aggregation row count: 513801
Unique collision_index in raw Vehicle table: 513801
Match: True

vehicle_type_diversity range: 1 to 6

Driver age sanity check:
       min_driver_age  max_driver_age  avg_driver_age
count   486161.000000   486161.000000   486161.000000
mean        35.518787       46.407184       40.931799
std         15.177365       17.160205       14.492094
min          1.000000        1.000000        1.000000
25%         24.000000       33.000000       30.000000
50%         32.000000       45.000000       39.000000
75%         44.000000       58.000000       50.000000
max        100.000000      105.000000      100.000000

Any duplicate collision_index in vehicle_agg? 0


8.5 Investigate Implausible Driver Age Values

min_driver_age shows a minimum of 1, which is not a realistic driver age. Following the same investigate-before-deciding approach used for the age_of_vehicle > 100 anomaly in the Data Quality notebook, inspect the affected records directly — including vehicle type — before deciding whether to treat, cap, or flag these values.

In [8]:
# Find vehicle records with implausibly low driver age
low_age_drivers = vehicle_df[vehicle_df["age_of_driver"] <= 5]

print("Records with age_of_driver <= 5:", len(low_age_drivers))
print("\nVehicle bucket breakdown for these records:")
print(low_age_drivers["vehicle_bucket"].value_counts())

print("\nSample records:")
display(low_age_drivers[["collision_index", "vehicle_type", "vehicle_bucket", "age_of_driver"]].head(15))

Records with age_of_driver <= 5: 198

Vehicle bucket breakdown for these records:
vehicle_bucket
pedal_cycle             166
other                    17
undocumented_code_33      7
car                       4
motorcycle                3
van_or_goods_light        1
Name: count, dtype: int64

Sample records:


,collision_index,vehicle_type,vehicle_bucket,age_of_driver
168675,2023131333270,90,other,1.0
494956,2024440294968,9,car,5.0
516941,2025440527959,9,car,4.0
592723,2022430137794,1,pedal_cycle,4.0
593124,2022201160643,90,other,5.0
593390,2022991208382,1,pedal_cycle,4.0
593916,2022471158726,1,pedal_cycle,5.0
594551,2022141170762,1,pedal_cycle,5.0
596241,2022991156843,1,pedal_cycle,5.0
598238,2022430242468,1,pedal_cycle,5.0


8.5 Investigate Implausible Driver Age Values — Findings

Of 198 records with age_of_driver ≤ 5, 166 (84%) are pedal cycles, where a young child riding a bicycle is a plausible real-world scenario — STATS19 records the cyclist as the vehicle's "driver." However, 4 car and 3 motorcycle records show the same implausible ages, which is physically impossible and indicates genuine data errors rather than valid observations.

Treatment decision: apply a vehicle-type-aware minimum age rule — retain low ages for pedal cycles, but convert age_of_driver to NaN for powered vehicle types (car, motorcycle, van, HGV, bus/coach) where the recorded age is below a plausible minimum (e.g., 10 years), since no legitimate scenario permits a young child operating a powered vehicle.

In [ ]:
# Define which buckets can plausibly have very young "drivers" (i.e., non-powered vehicles)
non_powered_buckets = ["pedal_cycle"]

# Flag implausible ages: young age AND a powered vehicle
implausible_age_mask = (
    (vehicle_df["age_of_driver"] < 10) &
    (~vehicle_df["vehicle_bucket"].isin(non_powered_buckets)) &
    (vehicle_df["age_of_driver"].notna())
)

print("Implausible young-age records in powered vehicles:", implausible_age_mask.sum())
display(vehicle_df.loc[implausible_age_mask, ["collision_index", "vehicle_type", "vehicle_bucket", "age_of_driver"]])

# Apply treatment: set these specific records' age_of_driver to NaN
vehicle_df.loc[implausible_age_mask, "age_of_driver"] = np.nan

print("\nRemaining age_of_driver < 10 in powered vehicles after treatment:",
      ((vehicle_df["age_of_driver"] < 10) & (~vehicle_df["vehicle_bucket"].isin(non_powered_buckets))).sum())

Implausible young-age records in powered vehicles: 176


,collision_index,vehicle_type,vehicle_bucket,age_of_driver
168675,2023131333270,90,other,1.0
494956,2024440294968,9,car,5.0
516941,2025440527959,9,car,4.0
593124,2022201160643,90,other,5.0
594991,2022111135405,2,motorcycle,7.0
...,...,...,...,...
925708,2023311308003,33,undocumented_code_33,9.0
926915,2023010448437,33,undocumented_code_33,9.0
928231,2023201298769,33,undocumented_code_33,9.0
931126,2023991331248,33,undocumented_code_33,6.0



Remaining age_of_driver < 10 in powered vehicles after treatment: 0


8.5 Refine Implausible-Age Treatment — Check Raw Vehicle Types Within Flagged Records

The "other" and "undocumented" buckets group together vehicle types with different plausibility profiles (e.g., ridden horse vs. agricultural vehicle vs. unknown). Before finalizing the treatment, break down the flagged records by their actual vehicle_type code to confirm the correction isn't incorrectly nulling out plausible young riders (e.g., horse riders) alongside genuine errors.

In [10]:
# Note: this re-inspects using vehicle_df's CURRENT state — since age_of_driver was already
# set to NaN for the 176 records, we check against the earlier captured mask/index instead
flagged_records = vehicle_df.loc[implausible_age_mask.reindex(vehicle_df.index, fill_value=False)] \
    if False else None  # placeholder — see note below

# Since age_of_driver is already NaN for these rows, re-identify them by index instead
flagged_indices = vehicle_df.index[implausible_age_mask]
flagged_vehicle_types = vehicle_df.loc[flagged_indices, "vehicle_type"].value_counts()

print("Raw vehicle_type breakdown of the 176 nulled records:")
print(flagged_vehicle_types)

Raw vehicle_type breakdown of the 176 nulled records:
vehicle_type
 33    84
 90    53
 23    10
 16     7
 97     7
 9      6
 2      4
 19     2
 22     1
-1      1
 17     1
Name: count, dtype: int64


8.5 Correct Over-Broad Treatment — Reload Original Ages and Reapply a Precise Rule

The initial treatment incorrectly assumed all non-pedal-cycle vehicle types were "powered" and therefore treated their young driver ages as errors. This wrongly included ridden horses, mobility scooters, and undocumented/unknown vehicle types where plausibility cannot be judged. Reload the original age_of_driver values from the cleaned CSV and reapply the correction using only vehicle types confirmed to require an adult operator: car, motorcycle (all variants), van/goods light, and agricultural vehicle.

In [11]:
# Reload original (untouched) age_of_driver values from the cleaned CSV
original_vehicle_df = pd.read_csv(data_path / "vehicle_cleaned.csv", dtype={"collision_index": str})

# Restore age_of_driver to its pre-correction state
vehicle_df["age_of_driver"] = original_vehicle_df["age_of_driver"]

# Define ONLY vehicle types confirmed to require an adult operator
confirmed_powered_types = [9, 108, 109,        # car (incl. historic/taxi)
                            2, 3, 4, 5, 23, 97, 103, 104, 105, 106,  # motorcycle variants
                            19,                 # van/goods light
                            17]                 # agricultural vehicle

implausible_age_mask_v2 = (
    (vehicle_df["age_of_driver"] < 10) &
    (vehicle_df["vehicle_type"].isin(confirmed_powered_types)) &
    (vehicle_df["age_of_driver"].notna())
)

print("Implausible young-age records (refined rule):", implausible_age_mask_v2.sum())
display(vehicle_df.loc[implausible_age_mask_v2, ["collision_index", "vehicle_type", "vehicle_bucket", "age_of_driver"]])

vehicle_df.loc[implausible_age_mask_v2, "age_of_driver"] = np.nan
print("\nTreatment applied. Records nulled:", implausible_age_mask_v2.sum())

Implausible young-age records (refined rule): 30


C:\Users\ACER\AppData\Local\Temp\ipykernel_18440\3096670442.py:2: DtypeWarning: Columns (27: generic_make_model) have mixed types. Specify dtype option on import or set low_memory=False.
  original_vehicle_df = pd.read_csv(data_path / "vehicle_cleaned.csv", dtype={"collision_index": str})


,collision_index,vehicle_type,vehicle_bucket,age_of_driver
494956,2024440294968,9,car,5.0
516941,2025440527959,9,car,4.0
594991,2022111135405,2,motorcycle,7.0
611151,2022101182461,19,van_or_goods_light,9.0
614977,2022131203787,23,motorcycle,5.0
624494,2022320214315,9,car,9.0
633315,202234NNC6942,9,car,5.0
756288,2024211480584,97,motorcycle,9.0
756344,2024311456647,9,car,9.0
766849,2024101428570,97,motorcycle,9.0



Treatment applied. Records nulled: 30


8.5 Implausible Driver Age — Final Treatment Applied

After refining the rule to only include vehicle types confirmed to require an adult operator (car, motorcycle variants, van/light goods, agricultural vehicle), 30 records were identified with age_of_driver under 10 and treated as data errors, converting age_of_driver to NaN. This is a substantial reduction from the initial over-broad rule (176 records), which incorrectly included ridden horses, mobility scooters, and undocumented/unknown vehicle types where plausibility could not be judged. The corrected vehicle_df now has age_of_driver restored to its original values except for these 30 confirmed-implausible records.

In [12]:
# Final verification
print("age_of_driver < 10 remaining (any vehicle type):", (vehicle_df["age_of_driver"] < 10).sum())
print("age_of_driver < 10 in pedal cycles (expected, retained):",
      ((vehicle_df["age_of_driver"] < 10) & (vehicle_df["vehicle_bucket"] == "pedal_cycle")).sum())
print("age_of_driver < 10 in confirmed powered vehicles (should be 0):",
      ((vehicle_df["age_of_driver"] < 10) & (vehicle_df["vehicle_type"].isin(confirmed_powered_types))).sum())

print("\nTotal NaN in age_of_driver now:", vehicle_df["age_of_driver"].isna().sum())

age_of_driver < 10 remaining (any vehicle type): 1257
age_of_driver < 10 in pedal cycles (expected, retained): 1111
age_of_driver < 10 in confirmed powered vehicles (should be 0): 0

Total NaN in age_of_driver now: 142270


8.6 Rebuild Vehicle Aggregation with Corrected Driver Ages

Re-run the Task 8.3 aggregation now that age_of_driver reflects the corrected values (implausible ages in confirmed-powered vehicles set to NaN, plausible young pedal-cyclists retained). This ensures min_driver_age, max_driver_age, and avg_driver_age in the final aggregation table are based on the validated data.

In [13]:
# Rebuild bucket dummies (unchanged, but redone for a clean full pipeline run)
bucket_dummies = pd.get_dummies(vehicle_df["vehicle_bucket"])
bucket_dummies["collision_index"] = vehicle_df["collision_index"]

vehicle_flags = bucket_dummies.groupby("collision_index").max().astype(int)
vehicle_flags.columns = [f"has_{col}" for col in vehicle_flags.columns]

vehicle_diversity = (
    vehicle_df.groupby("collision_index")["vehicle_bucket"]
    .nunique()
    .rename("vehicle_type_diversity")
)

# Driver age aggregates — now using corrected age_of_driver
driver_age_agg = (
    vehicle_df.groupby("collision_index")["age_of_driver"]
    .agg(min_driver_age="min", max_driver_age="max", avg_driver_age="mean")
)

vehicle_agg = vehicle_flags.join(vehicle_diversity).join(driver_age_agg).reset_index()

print("Vehicle aggregation shape:", vehicle_agg.shape)
print("Row count matches unique collisions:", vehicle_agg.shape[0] == vehicle_df["collision_index"].nunique())
print("\nUpdated driver age sanity check:")
print(vehicle_agg[["min_driver_age", "max_driver_age", "avg_driver_age"]].describe())

Vehicle aggregation shape: (513801, 13)
Row count matches unique collisions: True

Updated driver age sanity check:
       min_driver_age  max_driver_age  avg_driver_age
count   486150.000000   486150.000000   486150.000000
mean        35.520572       46.408094       40.933146
std         15.176285       17.159327       14.491189
min          1.000000        1.000000        1.000000
25%         24.000000       33.000000       30.000000
50%         32.000000       45.000000       39.000000
75%         44.000000       58.000000       50.000000
max        100.000000      105.000000      100.000000


8.6 Rebuild Vehicle Aggregation with Corrected Driver Ages

Re-run the Task 8.3 aggregation now that age_of_driver reflects the corrected values (implausible ages in confirmed-powered vehicles set to NaN, plausible young pedal-cyclists retained). This ensures min_driver_age, max_driver_age, and avg_driver_age in the final aggregation table are based on the validated data.

In [14]:
# Rebuild bucket dummies (unchanged, but redone for a clean full pipeline run)
bucket_dummies = pd.get_dummies(vehicle_df["vehicle_bucket"])
bucket_dummies["collision_index"] = vehicle_df["collision_index"]

vehicle_flags = bucket_dummies.groupby("collision_index").max().astype(int)
vehicle_flags.columns = [f"has_{col}" for col in vehicle_flags.columns]

vehicle_diversity = (
    vehicle_df.groupby("collision_index")["vehicle_bucket"]
    .nunique()
    .rename("vehicle_type_diversity")
)

# Driver age aggregates — now using corrected age_of_driver
driver_age_agg = (
    vehicle_df.groupby("collision_index")["age_of_driver"]
    .agg(min_driver_age="min", max_driver_age="max", avg_driver_age="mean")
)

vehicle_agg = vehicle_flags.join(vehicle_diversity).join(driver_age_agg).reset_index()

print("Vehicle aggregation shape:", vehicle_agg.shape)
print("Row count matches unique collisions:", vehicle_agg.shape[0] == vehicle_df["collision_index"].nunique())
print("\nUpdated driver age sanity check:")
print(vehicle_agg[["min_driver_age", "max_driver_age", "avg_driver_age"]].describe())

Vehicle aggregation shape: (513801, 13)
Row count matches unique collisions: True

Updated driver age sanity check:
       min_driver_age  max_driver_age  avg_driver_age
count   486150.000000   486150.000000   486150.000000
mean        35.520572       46.408094       40.933146
std         15.176285       17.159327       14.491189
min          1.000000        1.000000        1.000000
25%         24.000000       33.000000       30.000000
50%         32.000000       45.000000       39.000000
75%         44.000000       58.000000       50.000000
max        100.000000      105.000000      100.000000


8.7 Validate Vehicle Aggregation Against Collision's Recorded Vehicle Count

Compare the number of vehicles actually aggregated per collision against number_of_vehicles already recorded in the Collision table. This confirms the aggregation correctly captured every vehicle and flags any remaining mismatches for investigation (following the same count-consistency approach used in Notebook 05, Task 5).

In [15]:
# Actual vehicle count per collision from the raw Vehicle table
actual_vehicle_counts = (
    vehicle_df.groupby("collision_index")
    .size()
    .rename("actual_vehicle_count")
)

# Merge with Collision's recorded number_of_vehicles
vehicle_count_check = collision_df[["collision_index", "number_of_vehicles"]].merge(
    actual_vehicle_counts, on="collision_index", how="left"
)

vehicle_count_check["match"] = (
    vehicle_count_check["number_of_vehicles"] == vehicle_count_check["actual_vehicle_count"]
)

print("Total collisions checked:", len(vehicle_count_check))
print("Matching:", vehicle_count_check["match"].sum())
print("Mismatching:", (~vehicle_count_check["match"]).sum())
print("Missing actual_vehicle_count (no Vehicle records at all):", vehicle_count_check["actual_vehicle_count"].isna().sum())

print("\nMismatch percentage:", round((~vehicle_count_check["match"]).mean() * 100, 2), "%")

display(vehicle_count_check[~vehicle_count_check["match"]].head(10))

Total collisions checked: 513801
Matching: 513801
Mismatching: 0
Missing actual_vehicle_count (no Vehicle records at all): 0

Mismatch percentage: 0.0 %


,collision_index,number_of_vehicles,actual_vehicle_count,match


Task 8 — Complete

The Vehicle aggregation table (vehicle_agg) is validated: one row per collision, vehicle-type flags and diversity count derived from cleaned bucket data, driver-age aggregates corrected for implausible values, and a 100% match against Collision's recorded number_of_vehicles. This table is ready to be joined to the Collision base table in Phase 4.

In [16]:
casualty_class_counts = casualty_df["casualty_class"].value_counts(dropna=False).sort_index()
casualty_class_pct = casualty_df["casualty_class"].value_counts(normalize=True, dropna=False).sort_index() * 100

casualty_class_summary = pd.DataFrame({
    "Count": casualty_class_counts,
    "Percentage": casualty_class_pct.round(2)
})

print("Casualty Class distribution")
print("-" * 40)
display(casualty_class_summary)

Casualty Class distribution
----------------------------------------


,Count,Percentage
casualty_class,,
1,432883,66.31
2,125540,19.23
3,94398,14.46


9.1 Inspect Age Band Categories Before Defining Child/Elderly Thresholds

Before creating has_child_casualty and has_elderly_casualty flags, inspect the actual age_band_of_casualty categories and their official STATS19 definitions. Thresholds must be based on the dataset's own age bands rather than an arbitrary cutoff, per the project's documented requirement.

In [17]:
age_band_counts = casualty_df["age_band_of_casualty"].value_counts(dropna=False).sort_index()
age_band_pct = casualty_df["age_band_of_casualty"].value_counts(normalize=True, dropna=False).sort_index() * 100

age_band_summary = pd.DataFrame({
    "Count": age_band_counts,
    "Percentage": age_band_pct.round(2)
})

print("Age Band of Casualty distribution")
print("-" * 40)
display(age_band_summary)

Age Band of Casualty distribution
----------------------------------------


,Count,Percentage
age_band_of_casualty,,
-1,14106,2.16
1,10419,1.60
2,16597,2.54
3,30523,4.68
4,68000,10.42
5,70305,10.77
6,132804,20.34
7,100841,15.45
8,81997,12.56


9.1 Inspect Age Band Categories Before Defining Child/Elderly Thresholds

Before creating has_child_casualty and has_elderly_casualty flags, inspect the actual age_band_of_casualty categories and their official STATS19 definitions. Thresholds must be based on the dataset's own age bands rather than an arbitrary cutoff, per the project's documented requirement.

In [18]:
age_band_counts = casualty_df["age_band_of_casualty"].value_counts(dropna=False).sort_index()
age_band_pct = casualty_df["age_band_of_casualty"].value_counts(normalize=True, dropna=False).sort_index() * 100

age_band_summary = pd.DataFrame({
    "Count": age_band_counts,
    "Percentage": age_band_pct.round(2)
})

print("Age Band of Casualty distribution")
print("-" * 40)
display(age_band_summary)

Age Band of Casualty distribution
----------------------------------------


,Count,Percentage
age_band_of_casualty,,
-1,14106,2.16
1,10419,1.60
2,16597,2.54
3,30523,4.68
4,68000,10.42
5,70305,10.77
6,132804,20.34
7,100841,15.45
8,81997,12.56


9.2 Define Child and Elderly Casualty Thresholds

Age band definitions are taken directly from the official STATS19 data guide. Child is defined as age_band_of_casualty codes 1-3 (ages 0-15), aligning with the UK legal definition of a child and landing exactly on a band boundary. Elderly is defined as codes 10-11 (ages 66+), aligning with UK retirement-age convention and also landing on a clean band boundary. Records with code -1 (missing) are excluded from both flags rather than assumed to be non-child/non-elderly.

In [20]:
child_codes = [1, 2, 3]      # ages 0-15
elderly_codes = [10, 11]     # ages 66+

# Use pandas nullable "boolean" dtype instead of plain bool, so it can hold NaN
casualty_df["is_child_casualty"] = casualty_df["age_band_of_casualty"].isin(child_codes).astype("boolean")
casualty_df["is_elderly_casualty"] = casualty_df["age_band_of_casualty"].isin(elderly_codes).astype("boolean")

# Explicitly leave missing (-1) as neither — not assumed false
casualty_df.loc[casualty_df["age_band_of_casualty"] == -1, ["is_child_casualty", "is_elderly_casualty"]] = pd.NA

print("Child casualty records:", casualty_df["is_child_casualty"].sum())
print("Elderly casualty records:", casualty_df["is_elderly_casualty"].sum())
print("Missing age band (excluded from both):", (casualty_df["age_band_of_casualty"] == -1).sum())
print("\nDtype check:", casualty_df["is_child_casualty"].dtype)

Child casualty records: 57539
Elderly casualty records: 62858
Missing age band (excluded from both): 14106

Dtype check: boolean


9.3 Aggregate Casualty Flags to Collision Level

Group Casualty records by collision_index to produce collision-level descriptive flags: pedestrian_involved, has_passenger_casualty, has_child_casualty, has_elderly_casualty, and worst_casualty_severity. These are retained in the Master Dataset for EDA and descriptive analysis only — under the prospective/scenario-based prediction framing already agreed, they describe post-collision outcomes and must not be used as inputs to the primary severity classifier.

In [21]:
casualty_agg = casualty_df.groupby("collision_index").agg(
    pedestrian_involved=("casualty_class", lambda x: (x == 3).any()),
    has_passenger_casualty=("casualty_class", lambda x: (x == 2).any()),
    has_child_casualty=("is_child_casualty", lambda x: x.fillna(False).any()),
    has_elderly_casualty=("is_elderly_casualty", lambda x: x.fillna(False).any()),
    worst_casualty_severity=("casualty_severity", "min")  # 1=Fatal, 2=Serious, 3=Slight — min = most severe
).reset_index()

print("Casualty aggregation shape:", casualty_agg.shape)
print("Unique collision_index in raw Casualty table:", casualty_df["collision_index"].nunique())
print("Row count matches:", casualty_agg.shape[0] == casualty_df["collision_index"].nunique())

print("\nFlag summary:")
print(casualty_agg[["pedestrian_involved", "has_passenger_casualty", "has_child_casualty", "has_elderly_casualty"]].sum())

print("\nworst_casualty_severity distribution:")
print(casualty_agg["worst_casualty_severity"].value_counts().sort_index())

casualty_agg.head()

Casualty aggregation shape: (513801, 6)
Unique collision_index in raw Casualty table: 513801
Row count matches: True

Flag summary:
pedestrian_involved       90723
has_passenger_casualty    93670
has_child_casualty        50889
has_elderly_casualty      56448
dtype: Int64

worst_casualty_severity distribution:
worst_casualty_severity
1      7553
2    116813
3    389435
Name: count, dtype: int64


,collision_index,pedestrian_involved,has_passenger_casualty,has_child_casualty,has_elderly_casualty,worst_casualty_severity
0,2021010287148,False,False,False,False,3
1,2021010287149,False,True,False,False,2
2,2021010287151,False,True,False,False,2
3,2021010287155,True,False,False,False,1
4,2021010287157,False,False,False,False,3


9.4 Validate Casualty Aggregation Against Collision's Recorded Casualty Count

Compare the number of casualties actually present per collision (from the raw Casualty table) against number_of_casualties already recorded in the Collision table — completing the same count-consistency check performed for Vehicle in Task 8.7, as required by Task 9's checklist.

In [22]:
# Actual casualty count per collision from the raw Casualty table
actual_casualty_counts = (
    casualty_df.groupby("collision_index")
    .size()
    .rename("actual_casualty_count")
)

# Merge with Collision's recorded number_of_casualties
casualty_count_check = collision_df[["collision_index", "number_of_casualties"]].merge(
    actual_casualty_counts, on="collision_index", how="left"
)

casualty_count_check["match"] = (
    casualty_count_check["number_of_casualties"] == casualty_count_check["actual_casualty_count"]
)

print("Total collisions checked:", len(casualty_count_check))
print("Matching:", casualty_count_check["match"].sum())
print("Mismatching:", (~casualty_count_check["match"]).sum())
print("Missing actual_casualty_count (no Casualty records at all):", casualty_count_check["actual_casualty_count"].isna().sum())

print("\nMismatch percentage:", round((~casualty_count_check["match"]).mean() * 100, 2), "%")

display(casualty_count_check[~casualty_count_check["match"]].head(10))

Total collisions checked: 513801
Matching: 513801
Mismatching: 0
Missing actual_casualty_count (no Casualty records at all): 0

Mismatch percentage: 0.0 %


,collision_index,number_of_casualties,actual_casualty_count,match


Task 9 — Complete

The Casualty aggregation table (casualty_agg) is validated: one row per collision, descriptive flags built from documented age-band thresholds (child = ages 0-15, elderly = ages 66+, per official STATS19 bands), worst_casualty_severity computed as an EDA-only sanity field, and a 100% match against Collision's recorded number_of_casualties. All four descriptive flags plus worst_casualty_severity are explicitly excluded from the primary prospective severity model per the team's Prediction-Point Policy, and retained only for descriptive/EDA use.

Save Phase 3 Outputs — Vehicle and Casualty Aggregation Tables

Save the validated vehicle_agg and casualty_agg tables to the cleaned-data folder, so Phase 4 (Master Dataset construction) can load them directly without depending on this notebook's kernel state.

In [24]:
output_path = Path("C:/Users/ACER/Downloads/New folder/cleaned_datasets")
output_path.mkdir(parents=True, exist_ok=True)

vehicle_agg.to_csv(output_path / "vehicle_agg.csv", index=False)
casualty_agg.to_csv(output_path / "casualty_agg.csv", index=False)

print("Aggregation tables saved to:", output_path.resolve())
print("Vehicle aggregation:", vehicle_agg.shape)
print("Casualty aggregation:", casualty_agg.shape)

Aggregation tables saved to: C:\Users\ACER\Downloads\New folder\cleaned_datasets
Vehicle aggregation: (513801, 13)
Casualty aggregation: (513801, 6)
